# 📈 Sales & Demand Forecasting for Businesses
**Future Interns — Machine Learning Internship | Task 1**

---
### 🎯 Objective
Build a machine learning model to forecast future sales using historical business data.

### 📌 Dataset
**Superstore Sales Dataset** — A widely used retail dataset from Kaggle containing 4 years of sales transactions.

### 🔧 Workflow
1. Data Loading & Exploration
2. Data Cleaning & Preprocessing
3. Time-Based Feature Engineering
4. Forecasting Models (Linear Regression + Random Forest)
5. Model Evaluation & Error Analysis
6. Business-Ready Visualizations & Insights

---

## ⚙️ Step 0 — Install & Import Libraries

In [ ]:
# Install required packages (run once)
# !pip install pandas numpy matplotlib seaborn scikit-learn openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

# Plot style
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')
PALETTE = ['#4361ee', '#f72585', '#4cc9f0', '#7209b7', '#3a0ca3']

print('✅ All libraries imported successfully!')

## 📥 Step 1 — Load Dataset

> **Dataset Source:** [Superstore Sales — Kaggle](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)  
> Download `Sample - Superstore.csv` and place it in the same folder as this notebook.

In [ ]:
# ── Load Data ──────────────────────────────────────────────────────────────
df = pd.read_csv('Sample - Superstore.csv', encoding='latin-1')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
df.head()

## 🔍 Step 2 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Basic Info ──────────────────────────────────────────────────────────────
print('── Data Types ──')
print(df.dtypes)
print('\n── Missing Values ──')
print(df.isnull().sum())
print('\n── Summary Statistics ──')
df[['Sales', 'Profit', 'Discount', 'Quantity']].describe().round(2)

In [ ]:
# ── Sales Distribution by Category ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Sales Distribution Overview', fontsize=16, fontweight='bold', y=1.02)

# By Category
cat_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=True)
axes[0].barh(cat_sales.index, cat_sales.values, color=PALETTE[:3])
axes[0].set_title('Total Sales by Category', fontweight='bold')
axes[0].set_xlabel('Total Sales ($)')
for i, v in enumerate(cat_sales.values):
    axes[0].text(v + 1000, i, f'${v:,.0f}', va='center', fontsize=9)

# By Region
reg_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
axes[1].bar(reg_sales.index, reg_sales.values, color=PALETTE)
axes[1].set_title('Total Sales by Region', fontweight='bold')
axes[1].set_ylabel('Total Sales ($)')
for i, v in enumerate(reg_sales.values):
    axes[1].text(i, v + 1000, f'${v:,.0f}', ha='center', fontsize=9)

# Sales vs Profit scatter
axes[2].scatter(df['Sales'], df['Profit'], alpha=0.3, color=PALETTE[0], s=15)
axes[2].axhline(0, color='red', linestyle='--', linewidth=1)
axes[2].set_title('Sales vs Profit', fontweight='bold')
axes[2].set_xlabel('Sales ($)')
axes[2].set_ylabel('Profit ($)')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA chart saved.')

## 🧹 Step 3 — Data Cleaning & Preprocessing

In [ ]:
# ── Parse Dates ─────────────────────────────────────────────────────────────
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

# ── Drop duplicates ──────────────────────────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Removed {before - len(df)} duplicate rows.')

# ── Aggregate to monthly sales (time-series target) ──────────────────────────
df_monthly = (
    df.groupby(pd.Grouper(key='Order Date', freq='ME'))
    ['Sales'].sum()
    .reset_index()
    .rename(columns={'Order Date': 'Month', 'Sales': 'Monthly_Sales'})
)

print(f'\nMonthly time-series shape: {df_monthly.shape}')
df_monthly.tail(10)

## 🏗️ Step 4 — Time-Based Feature Engineering

In [ ]:
# ── Extract temporal features ────────────────────────────────────────────────
df_monthly['Year']        = df_monthly['Month'].dt.year
df_monthly['MonthNum']    = df_monthly['Month'].dt.month
df_monthly['Quarter']     = df_monthly['Month'].dt.quarter
df_monthly['TimeIndex']   = range(len(df_monthly))   # linear time trend

# Cyclical encoding of month (captures seasonality)
df_monthly['Month_sin'] = np.sin(2 * np.pi * df_monthly['MonthNum'] / 12)
df_monthly['Month_cos'] = np.cos(2 * np.pi * df_monthly['MonthNum'] / 12)

# Lag features (previous months)
df_monthly['Lag_1']  = df_monthly['Monthly_Sales'].shift(1)
df_monthly['Lag_2']  = df_monthly['Monthly_Sales'].shift(2)
df_monthly['Lag_3']  = df_monthly['Monthly_Sales'].shift(3)
df_monthly['Lag_12'] = df_monthly['Monthly_Sales'].shift(12)  # same month last year

# Rolling statistics
df_monthly['Rolling_3m_mean'] = df_monthly['Monthly_Sales'].shift(1).rolling(3).mean()
df_monthly['Rolling_6m_mean'] = df_monthly['Monthly_Sales'].shift(1).rolling(6).mean()
df_monthly['Rolling_3m_std']  = df_monthly['Monthly_Sales'].shift(1).rolling(3).std()

# Drop NaN rows created by lag features
df_model = df_monthly.dropna().copy()

print(f'Model-ready rows: {len(df_model)}')
print('Features created:', [c for c in df_model.columns if c not in ['Month','Monthly_Sales']])

## 🤖 Step 5 — Model Training

In [ ]:
# ── Define Features & Target ─────────────────────────────────────────────────
FEATURES = ['TimeIndex', 'Year', 'MonthNum', 'Quarter',
            'Month_sin', 'Month_cos',
            'Lag_1', 'Lag_2', 'Lag_3', 'Lag_12',
            'Rolling_3m_mean', 'Rolling_6m_mean', 'Rolling_3m_std']

TARGET = 'Monthly_Sales'

X = df_model[FEATURES]
y = df_model[TARGET]

# ── Chronological Train/Test Split (no shuffling for time series!) ────────────
split_idx = int(len(X) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = df_model['Month'].iloc[split_idx:]

print(f'Train size: {len(X_train)} months | Test size: {len(X_test)} months')

# ── Train Models ─────────────────────────────────────────────────────────────
models = {
    'Linear Regression':    LinearRegression(),
    'Random Forest':        RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42),
    'Gradient Boosting':    GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
}

results = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    predictions[name] = preds
    
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    mape = np.mean(np.abs((y_test.values - preds) / y_test.values)) * 100
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R²': r2, 'MAPE (%)': mape}
    print(f'{name:25s} | MAE=${mae:,.0f} | RMSE=${rmse:,.0f} | R²={r2:.3f} | MAPE={mape:.1f}%')

print('\n✅ All models trained.')

## 📊 Step 6 — Model Evaluation & Error Analysis

In [ ]:
# ── Metrics Table ────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results).T.round(2)
results_df.index.name = 'Model'
print('\n📋 Model Comparison:')
display(results_df.style
        .highlight_min(subset=['MAE','RMSE','MAPE (%)'], color='#c8f5c0')
        .highlight_max(subset=['R²'], color='#c8f5c0')
        .format({'MAE': '${:,.0f}', 'RMSE': '${:,.0f}', 'R²': '{:.3f}', 'MAPE (%)': '{:.1f}%'}))

In [ ]:
# ── Forecast vs Actual Plot ──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 14))
fig.suptitle('Forecast vs Actual Sales — All Models', fontsize=16, fontweight='bold')

for ax, (name, preds) in zip(axes, predictions.items()):
    ax.plot(dates_test.values, y_test.values, label='Actual', color='#1a1a2e', linewidth=2.5)
    ax.plot(dates_test.values, preds, label=f'Predicted ({name})', color=PALETTE[0],
            linewidth=2, linestyle='--')
    ax.fill_between(dates_test.values, y_test.values, preds, alpha=0.15, color=PALETTE[1])
    ax.set_title(f'{name}  |  MAPE={results[name]["MAPE (%)"]:.1f}%  |  R²={results[name]["R²"]:.3f}',
                 fontweight='bold')
    ax.set_ylabel('Monthly Sales ($)')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.savefig('forecast_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as forecast_vs_actual.png')

In [ ]:
# ── Residual Analysis (Best Model: Gradient Boosting) ────────────────────────
best_model_name = min(results, key=lambda m: results[m]['MAPE (%)'])
best_preds = predictions[best_model_name]
residuals = y_test.values - best_preds

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Residual Analysis — {best_model_name}', fontsize=14, fontweight='bold')

# Residuals over time
axes[0].bar(range(len(residuals)), residuals, color=[
    PALETTE[0] if r >= 0 else PALETTE[1] for r in residuals])
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_title('Residuals Over Time')
axes[0].set_xlabel('Test Period (months)')
axes[0].set_ylabel('Residual ($)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Residual distribution
axes[1].hist(residuals, bins=12, color=PALETTE[0], edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual ($)')
axes[1].set_ylabel('Count')

# Predicted vs Actual scatter
axes[2].scatter(y_test.values, best_preds, color=PALETTE[0], alpha=0.7, s=60, edgecolors='white')
lims = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
axes[2].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Fit')
axes[2].set_title('Predicted vs Actual')
axes[2].set_xlabel('Actual Sales ($)')
axes[2].set_ylabel('Predicted Sales ($)')
axes[2].legend()
axes[2].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Residual analysis saved.')

In [ ]:
# ── Feature Importance (Random Forest) ───────────────────────────────────────
rf_model = models['Random Forest']
feat_imp = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [PALETTE[0] if v > feat_imp.median() else PALETTE[2] for v in feat_imp.values]
feat_imp.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.axvline(feat_imp.median(), color='red', linestyle='--', linewidth=1, label='Median')
ax.legend()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance chart saved.')

## 🔮 Step 7 — Future Forecast (Next 6 Months)

In [ ]:
# ── Generate future dates & iteratively forecast ─────────────────────────────
best_model = models[best_model_name]
last_row   = df_model.iloc[-1].copy()
history    = df_model['Monthly_Sales'].tolist()

future_months = 6
future_preds  = []
future_dates  = pd.date_range(df_model['Month'].max(), periods=future_months + 1, freq='ME')[1:]

for i, date in enumerate(future_dates):
    t_idx     = df_model['TimeIndex'].max() + i + 1
    month_num = date.month
    feat_row  = pd.DataFrame([{
        'TimeIndex':        t_idx,
        'Year':             date.year,
        'MonthNum':         month_num,
        'Quarter':          date.quarter,
        'Month_sin':        np.sin(2 * np.pi * month_num / 12),
        'Month_cos':        np.cos(2 * np.pi * month_num / 12),
        'Lag_1':            history[-1],
        'Lag_2':            history[-2],
        'Lag_3':            history[-3],
        'Lag_12':           history[-12],
        'Rolling_3m_mean':  np.mean(history[-3:]),
        'Rolling_6m_mean':  np.mean(history[-6:]),
        'Rolling_3m_std':   np.std(history[-3:]),
    }])
    pred = best_model.predict(feat_row[FEATURES])[0]
    future_preds.append(pred)
    history.append(pred)

future_df = pd.DataFrame({'Month': future_dates, 'Forecasted_Sales': future_preds})
print(f'\n📅 {future_months}-Month Sales Forecast using {best_model_name}:')
print(future_df.to_string(index=False, float_format='${:,.0f}'.format))

In [ ]:
# ── Business-Ready Forecast Chart ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 6))

# Historical
ax.plot(df_monthly['Month'], df_monthly['Monthly_Sales'],
        label='Historical Sales', color='#1a1a2e', linewidth=2)

# Test predictions overlay
ax.plot(dates_test.values, best_preds,
        label=f'Model Fit ({best_model_name})', color=PALETTE[0],
        linewidth=1.5, linestyle='--', alpha=0.8)

# Future forecast
ax.plot(future_df['Month'], future_df['Forecasted_Sales'],
        label='Forecast (Next 6 Months)', color=PALETTE[1],
        linewidth=2.5, linestyle='-.', marker='o', markersize=7)

# Shade forecast region
ax.axvspan(future_df['Month'].min(), future_df['Month'].max(),
           alpha=0.08, color=PALETTE[1], label='Forecast Window')

# Annotations for forecast values
for _, row in future_df.iterrows():
    ax.annotate(f"${row['Forecasted_Sales']:,.0f}",
                xy=(row['Month'], row['Forecasted_Sales']),
                xytext=(0, 12), textcoords='offset points',
                ha='center', fontsize=8, color=PALETTE[1], fontweight='bold')

ax.set_title('📈 Sales Forecast Dashboard — Future Interns ML Task 1',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Month')
ax.set_ylabel('Monthly Sales ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.legend(loc='upper left', framealpha=0.9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sales_forecast_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Final forecast dashboard saved as sales_forecast_dashboard.png')

## 💼 Step 8 — Business Insights & Recommendations

In [ ]:
# ── Seasonality Heatmap ───────────────────────────────────────────────────────
pivot = df_monthly.copy()
pivot['Year']     = pivot['Month'].dt.year
pivot['MonthNum'] = pivot['Month'].dt.month
heatmap_data = pivot.pivot(index='Year', columns='MonthNum', values='Monthly_Sales')
heatmap_data.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                         'Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Monthly Sales ($)'})
ax.set_title('🗓️ Seasonality Heatmap — Monthly Sales by Year',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Year')
plt.tight_layout()
plt.savefig('seasonality_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Seasonality heatmap saved.')

In [ ]:
# ── Summary Business Report ───────────────────────────────────────────────────
total_sales    = df['Sales'].sum()
avg_monthly    = df_monthly['Monthly_Sales'].mean()
peak_month     = df_monthly.loc[df_monthly['Monthly_Sales'].idxmax()]
low_month      = df_monthly.loc[df_monthly['Monthly_Sales'].idxmin()]
forecast_total = future_df['Forecasted_Sales'].sum()

print('='*60)
print('       📊 BUSINESS INSIGHTS REPORT')
print('='*60)
print(f'  Total Historical Sales   : ${total_sales:>12,.0f}')
print(f'  Average Monthly Sales    : ${avg_monthly:>12,.0f}')
print(f'  Peak Month               : {peak_month["Month"].strftime("%B %Y")} — ${peak_month["Monthly_Sales"]:,.0f}')
print(f'  Lowest Month             : {low_month["Month"].strftime("%B %Y")} — ${low_month["Monthly_Sales"]:,.0f}')
print(f'  Best Forecast Model      : {best_model_name}')
print(f'  Model MAPE               : {results[best_model_name]["MAPE (%)"]:.1f}%')
print(f'  Model R²                 : {results[best_model_name]["R²"]:.3f}')
print(f'  Forecast (Next 6 Months) : ${forecast_total:>12,.0f}')
print('='*60)
print()
print('📌 KEY RECOMMENDATIONS')
print('  1. Sales peak in Q4 — increase inventory & marketing budget.')
print('  2. Weak months (Q1) are ideal for promotions & discount campaigns.')
print('  3. Lag-1 (last month sales) is the strongest predictor — monitor closely.')
print('  4. Use forecasts to plan supply chain 2-3 months in advance.')
print()
print('✅ Full project complete. All charts saved in working directory.')

---
## ✅ Project Summary

| Step | Description | Output |
|------|-------------|--------|
| 1 | Data Loading & EDA | `eda_overview.png` |
| 2 | Cleaning & Preprocessing | Clean DataFrame |
| 3 | Feature Engineering | 13 temporal features |
| 4 | Model Training | Linear Reg, RF, GBR |
| 5 | Evaluation | MAE, RMSE, R², MAPE |
| 6 | Residual Analysis | `residual_analysis.png` |
| 7 | Future Forecast | `sales_forecast_dashboard.png` |
| 8 | Business Insights | Seasonality + Recommendations |

**GitHub Repo:** `FUTURE_ML_01`  
**Author:** *(your name)*  
**Internship:** Future Interns — Machine Learning Track